# 🍅 Train YOLO11 - Tomato Disease Detection (Google Colab)

Fine-tune **YOLO11s** trên dataset **10 loại bệnh cà chua** với GPU mạnh hơn trên Colab.

| Colab GPU | VRAM | Batch khuyến nghị |
|-----------|------|-------------------|
| T4 (Free) | 15 GB | 32 |
| A100 (Pro) | 40 GB | 64 |
| L4 (Pro) | 24 GB | 48 |

**Thời gian dự kiến**: ~30-60 phút (T4), ~15-20 phút (A100)

## 📋 Bước 0: Kiểm tra GPU

In [1]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/bin/bash: line 1: nvidia-smi: command not found

PyTorch: 2.10.0+cpu
CUDA: False


## 📦 Bước 1: Cài đặt Ultralytics

In [ ]:
!pip install -q ultralytics

## 📂 Bước 2: Upload dataset lên Google Drive

### Cách chuẩn bị trên máy local:

**Trên máy Windows**, nén thư mục dataset:
```
ai-service/dataforYolo/tomato_detection/
├── train/images/  (8,546 ảnh)
├── train/labels/  (8,546 file .txt)
├── valid/images/  (1,062 ảnh)
├── valid/labels/  (1,062 file .txt)
├── test/images/   (1,079 ảnh)
├── test/labels/   (1,079 file .txt)
└── data.yaml
```

Mở PowerShell và chạy:
```powershell
Compress-Archive -Path "D:\DoAn_Garden\smart_garden\ai-service\dataforYolo\tomato_detection" -DestinationPath "D:\DoAn_Garden\smart_garden\tomato_detection.zip"
```

Upload file `tomato_detection.zip` lên **Google Drive** (thư mục gốc hoặc bất kỳ).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ============================
# 🔧 CHỈNH SỬA ĐƯỜNG DẪN NÀY
# ============================
# Đường dẫn tới file zip trên Google Drive
ZIP_PATH = "/content/drive/MyDrive/tomato_detection.zip"

# Thư mục làm việc trên Colab
WORK_DIR = "/content/tomato_yolo"
DATA_DIR = f"{WORK_DIR}/tomato_detection"

os.makedirs(WORK_DIR, exist_ok=True)

# Giải nén
if not os.path.exists(DATA_DIR):
    print(f"Giải nén {ZIP_PATH}...")
    !unzip -q "{ZIP_PATH}" -d "{WORK_DIR}"
    print("Xong!")
else:
    print(f"Dataset đã tồn tại tại {DATA_DIR}")

# Kiểm tra
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATA_DIR, split, 'images')
    n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f"  {split}: {n} images")

## 🔧 Bước 3: Cập nhật data.yaml (đường dẫn Colab)

In [ ]:
import yaml

yaml_path = f"{DATA_DIR}/data.yaml"

data_config = {
    "path": DATA_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 10,
    "names": [
        "Tomato Bacterial Spot",
        "Tomato Early blight",
        "Tomato Late blight",
        "Tomato Leaf Mold",
        "Tomato Septoria leaf spot",
        "Tomato Spider mites Two-spotted spider mite",
        "Tomato Target Spot",
        "Tomato Yellow Leaf Curl Virus",
        "Tomato healthy",
        "Tomato mosaic virus"
    ]
}

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, allow_unicode=True)

print(f"data.yaml đã cập nhật:")
!cat "{yaml_path}"

## 🚀 Bước 4: Train YOLO11

In [ ]:
from ultralytics import YOLO
import torch

# ============================
# 🔧 CẤU HÌNH TRAINING
# ============================
MODEL_NAME = "yolo11s.pt"       # yolo11n.pt (nhỏ) | yolo11s.pt (vừa) | yolo11m.pt (lớn)
EPOCHS = 100
PATIENCE = 20                    # Early stopping
IMGSZ = 640

# Tự chọn batch size theo GPU
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if vram_gb >= 35:      # A100
    BATCH = 64
elif vram_gb >= 20:    # L4 / A10
    BATCH = 48
elif vram_gb >= 14:    # T4
    BATCH = 32
elif vram_gb >= 8:     # P100
    BATCH = 16
else:
    BATCH = 8

print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
print(f"Model: {MODEL_NAME} | Batch: {BATCH} | Epochs: {EPOCHS}")
print("=" * 60)

In [ ]:
# Bắt đầu training
model = YOLO(MODEL_NAME)

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    workers=4,
    project=f"{WORK_DIR}/runs",
    name="yolo11s_tomato",
    exist_ok=True,
    # ── Optimization ──
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    # ── Early stopping ──
    patience=PATIENCE,
    # ── Augmentation ──
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    # ── Loss weights ──
    box=7.5,
    cls=1.5,
    dfl=1.5,
    # ── Saving ──
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

print("\n" + "=" * 60)
print("TRAINING XONG!")
print("=" * 60)

## 📊 Bước 5: Đánh giá trên Test Set

In [ ]:
from ultralytics import YOLO

best_pt = f"{WORK_DIR}/runs/yolo11s_tomato/weights/best.pt"
best_model = YOLO(best_pt)

metrics = best_model.val(
    data=yaml_path,
    split="test",
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    plots=True,
    verbose=True,
)

print(f"\n{'='*60}")
print(f"  Test mAP50:    {metrics.box.map50:.4f}")
print(f"  Test mAP50-95: {metrics.box.map:.4f}")
print(f"{'='*60}")

## 📈 Bước 6: Xem kết quả training

In [ ]:
from IPython.display import Image, display
import os

run_dir = f"{WORK_DIR}/runs/yolo11s_tomato"

plots = ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
         'F1_curve.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png',
         'labels.jpg', 'labels_correlogram.jpg']

for p in plots:
    path = os.path.join(run_dir, p)
    if os.path.exists(path):
        print(f"\n{'─'*40} {p} {'─'*40}")
        display(Image(filename=path, width=800))

## 💾 Bước 7: Lưu model về Google Drive & Download

In [ ]:
import shutil

# Lưu toàn bộ kết quả vào Drive
drive_dest = "/content/drive/MyDrive/yolo11_tomato_results"
os.makedirs(drive_dest, exist_ok=True)

# Copy weights
best_src = f"{WORK_DIR}/runs/yolo11s_tomato/weights/best.pt"
last_src = f"{WORK_DIR}/runs/yolo11s_tomato/weights/last.pt"

shutil.copy2(best_src, f"{drive_dest}/best.pt")
shutil.copy2(last_src, f"{drive_dest}/last.pt")

# Copy plots
for p in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
          'F1_curve.png', 'PR_curve.png', 'results.csv']:
    src = f"{WORK_DIR}/runs/yolo11s_tomato/{p}"
    if os.path.exists(src):
        shutil.copy2(src, f"{drive_dest}/{p}")

print(f"Đã lưu vào: {drive_dest}")
print(f"  best.pt: {os.path.getsize(best_src) / 1e6:.1f} MB")
!ls -la "{drive_dest}"

In [ ]:
# Download trực tiếp best.pt về máy local
from google.colab import files
files.download(best_src)

## 🔗 Bước 8: Tích hợp vào project (trên máy local)

Sau khi download `best.pt`, copy vào project:

```powershell
# PowerShell trên Windows
Copy-Item "$HOME\Downloads\best.pt" "D:\DoAn_Garden\smart_garden\ai-service\app\ml\models\plant\plant_detect\weights\best.pt" -Force
```

`PlantImageService` sẽ tự động load model mới khi nhận request tiếp theo.

## 🧪 (Tùy chọn) Test nhanh trên 1 ảnh

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob

model = YOLO(f"{WORK_DIR}/runs/yolo11s_tomato/weights/best.pt")

# Lấy 1 ảnh từ test set
test_imgs = glob.glob(f"{DATA_DIR}/test/images/*.jpg")[:4]

results = model.predict(
    source=test_imgs,
    imgsz=640,
    conf=0.25,
    save=True,
    project=f"{WORK_DIR}/predictions",
    name="test_samples",
    exist_ok=True,
)

# Hiển thị
for img_path in glob.glob(f"{WORK_DIR}/predictions/test_samples/*.jpg"):
    display(Image(filename=img_path, width=500))